# Library schema and Alembic, step by step

Two schemas, built for real:

1. **the library** — the mini-project for these four weeks
2. **Harbour** — the main project

Then the same schemas under **Alembic**, so every change gets a version.

## Setup

Run once, in the VS Code terminal.

**1. Postgres** — skip if `psql --version` already answers.

```bash
sudo install -d /usr/share/postgresql-common/pgdg
sudo curl -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc \
  https://www.postgresql.org/media/keys/ACCC4CF8.asc
echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.asc] http://apt.postgresql.org/pub/repos/apt $(lsb_release -cs)-pgdg main" \
  | sudo tee /etc/apt/sources.list.d/pgdg.list
sudo apt update
sudo apt install postgresql-18 postgresql-client-18 libpq5
```

**2. Python packages**

```bash
uv venv && source .venv/bin/activate
uv pip install "psycopg>=3,<4" alembic sqlalchemy
```

**3. Server, role, database**

```bash
sudo pg_ctlcluster 18 main start
sudo -u postgres createuser -s gamp
sudo -u postgres createdb library -O gamp
```

**4. VS Code** — `Ctrl+Shift+P` -> Python: Select Interpreter -> `.venv/bin/python`, then Developer: Reload Window.

When you are done: `dropdb library`. Deleting `.venv` does not touch the database.

In [ ]:
import os

import psycopg

# Local Postgres over its Unix socket: no password needed.
DEFAULT_URL = "postgresql:///library?host=/var/run/postgresql"
DATABASE_URL = os.environ.get("DATABASE_URL", DEFAULT_URL)


def q(sql, params=None, fetch=False):
    with psycopg.connect(DATABASE_URL, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            if fetch and cur.description:
                return cur.fetchall()
            return None


try:
    print("connected:", q("SELECT version()", fetch=True)[0][0].split(",")[0])
except Exception as exc:
    print("Could not connect ->", exc)

## Reset

So the notebook can be run twice. `CASCADE` drops anything pointing at these tables.

In [ ]:
for table in ("fines", "loans", "copies", "members", "books"):
    q(f"DROP TABLE IF EXISTS {table} CASCADE")

print("clean slate")

## The library: five tables

A **book** is the title. A **copy** is the physical thing on the shelf. Two copies of the same book are two rows in `copies`, one row in `books` — that is why there are five tables and not four.

In [ ]:
q("""
CREATE TABLE books (
    book_id     TEXT PRIMARY KEY,
    title       TEXT NOT NULL,
    author      TEXT NOT NULL,
    isbn        TEXT UNIQUE,
    created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
)
""")

In [ ]:
q("""
CREATE TABLE members (
    member_id  TEXT PRIMARY KEY,
    full_name  TEXT NOT NULL,
    email      TEXT NOT NULL UNIQUE,
    joined_on  DATE NOT NULL DEFAULT CURRENT_DATE
)
""")

In [ ]:
q("""
CREATE TABLE copies (
    copy_id  SERIAL PRIMARY KEY,
    book_id  TEXT NOT NULL REFERENCES books(book_id),
    shelf    TEXT NOT NULL,
    status   TEXT NOT NULL DEFAULT 'available'
             CHECK (status IN ('available', 'on_loan', 'lost'))
)
""")

In [ ]:
q("""
CREATE TABLE loans (
    loan_id      SERIAL PRIMARY KEY,
    copy_id      INTEGER NOT NULL REFERENCES copies(copy_id),
    member_id    TEXT NOT NULL REFERENCES members(member_id),
    borrowed_on  DATE NOT NULL DEFAULT CURRENT_DATE,
    due_on       DATE NOT NULL,
    returned_on  DATE,
    CHECK (due_on >= borrowed_on)
)
""")

### The one rule SQL cannot express with a CHECK

*A copy can have only one open loan.* A `CHECK` sees one row at a time, so it cannot count. A **partial unique index** can — unique, but only over rows where `returned_on IS NULL`.

In [ ]:
q("""
CREATE UNIQUE INDEX one_open_loan_per_copy
    ON loans (copy_id)
    WHERE returned_on IS NULL
""")

In [ ]:
q("""
CREATE TABLE fines (
    fine_id      SERIAL PRIMARY KEY,
    loan_id      INTEGER NOT NULL REFERENCES loans(loan_id),
    amount_cents INTEGER NOT NULL CHECK (amount_cents >= 0),
    paid_on      DATE
)
""")

In [ ]:
rows = q("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
""", fetch=True)

for (name,) in rows:
    print(name)

## Library schema v1

### books

| Column | Type | Rule |
|---|---|---|
| `book_id` | TEXT | primary key, e.g. `BK-001` |
| `title` | TEXT | not null |
| `author` | TEXT | not null |
| `isbn` | TEXT | unique, optional |
| `created_at` | TIMESTAMPTZ | set by the database |

### members

| Column | Type | Rule |
|---|---|---|
| `member_id` | TEXT | primary key, e.g. `MB-01` |
| `full_name` | TEXT | not null |
| `email` | TEXT | unique, not null |
| `joined_on` | DATE | today, by default |

### copies

| Column | Type | Rule |
|---|---|---|
| `copy_id` | SERIAL | primary key |
| `book_id` | TEXT | foreign key -> `books.book_id` |
| `shelf` | TEXT | not null |
| `status` | TEXT | `available` / `on_loan` / `lost`, defaults to `available` |

### loans

| Column | Type | Rule |
|---|---|---|
| `loan_id` | SERIAL | primary key |
| `copy_id` | INTEGER | foreign key -> `copies.copy_id` |
| `member_id` | TEXT | foreign key -> `members.member_id` |
| `borrowed_on` | DATE | today, by default |
| `due_on` | DATE | not null, on or after `borrowed_on` |
| `returned_on` | DATE | empty while the loan is open |

### fines

| Column | Type | Rule |
|---|---|---|
| `fine_id` | SERIAL | primary key |
| `loan_id` | INTEGER | foreign key -> `loans.loan_id` |
| `amount_cents` | INTEGER | money in cents, never negative |
| `paid_on` | DATE | empty until paid |

### How they connect

| From | To | Relationship |
|---|---|---|
| `books` | `copies` | one book, many copies |
| `copies` | `loans` | one copy, many loans over time — one open at a time |
| `members` | `loans` | one member, many loans |
| `loans` | `fines` | one loan, many fines |

## Does it hold?

Real data, then three rows the schema should refuse.

In [ ]:
q("INSERT INTO books (book_id, title, author, isbn) VALUES (%s, %s, %s, %s)",
  ("BK-001", "The Salt Road", "Ngozi Eze", "978-1-4028-9462-6"))
q("INSERT INTO books (book_id, title, author, isbn) VALUES (%s, %s, %s, %s)",
  ("BK-002", "Quiet Machines", "Tunde Bello", None))

q("INSERT INTO members (member_id, full_name, email) VALUES (%s, %s, %s)",
  ("MB-01", "Ada Obi", "ada@example.com"))
q("INSERT INTO members (member_id, full_name, email) VALUES (%s, %s, %s)",
  ("MB-02", "Femi Ade", "femi@example.com"))

# two copies of the same title: one book row, two copy rows
for book_id, shelf in [("BK-001", "A1"), ("BK-001", "A2"), ("BK-002", "B3")]:
    q("INSERT INTO copies (book_id, shelf) VALUES (%s, %s)", (book_id, shelf))

for row in q("SELECT copy_id, book_id, shelf, status FROM copies ORDER BY copy_id", fetch=True):
    print(row)

In [ ]:
# copy 1 is out now
q("INSERT INTO loans (copy_id, member_id, due_on) VALUES (1, 'MB-01', CURRENT_DATE + 14)")

# copy 2 came back four days late
q("""
   INSERT INTO loans (copy_id, member_id, borrowed_on, due_on, returned_on)
   VALUES (2, 'MB-01', CURRENT_DATE - 30, CURRENT_DATE - 16, CURRENT_DATE - 20)
""")
q("INSERT INTO fines (loan_id, amount_cents) VALUES (2, %s)", (4 * 250,))

print(q("SELECT loan_id, copy_id, member_id, due_on, returned_on FROM loans", fetch=True))
print(q("SELECT loan_id, amount_cents AS naira_worth_250_each FROM fines", fetch=True))

In [ ]:
bad = [
    ("same copy lent twice", "INSERT INTO loans (copy_id, member_id, due_on) VALUES (1, 'MB-02', CURRENT_DATE + 14)"),
    ("due before borrowed", "INSERT INTO loans (copy_id, member_id, borrowed_on, due_on) VALUES (3, 'MB-02', CURRENT_DATE, CURRENT_DATE - 1)"),
    ("book that does not exist", "INSERT INTO copies (book_id, shelf) VALUES ('BK-999', 'Z9')"),
    ("two members, one email", "INSERT INTO members (member_id, full_name, email) VALUES ('MB-03', 'Third One', 'ada@example.com')"),
    ("negative fine", "INSERT INTO fines (loan_id, amount_cents) VALUES (1, -500)"),
    ("impossible status", "INSERT INTO copies (book_id, shelf, status) VALUES ('BK-002', 'C1', 'missing')"),
]

for label, statement in bad:
    try:
        q(statement)
        print("ACCEPTED  <- wrong:", label)
    except Exception as exc:
        print("rejected :", label, "|", str(exc).splitlines()[0])

In [ ]:
# the copy came back today, so it can go out again
q("UPDATE loans SET returned_on = CURRENT_DATE WHERE loan_id = 1")
q("INSERT INTO loans (copy_id, member_id, due_on) VALUES (1, 'MB-02', CURRENT_DATE + 14)")

print(q("SELECT copy_id, member_id, returned_on FROM loans WHERE copy_id = 1", fetch=True))

## Alembic

Those `CREATE TABLE` statements only exist in this notebook. Alembic gives a schema the same treatment git gives code: every change is a numbered file you can replay, review, and undo.

Three commands do most of the work:

| Command | What it does |
|---|---|
| `alembic revision --autogenerate -m "message"` | compares your models to the database, writes a new revision file |
| `alembic upgrade head` | applies every revision the database has not seen |
| `alembic current` | prints the revision the database is on |

## One-time setup

```bash
mkdir -p ~/harbour/alembic_demo && cd ~/harbour/alembic_demo
alembic init migrations
```

That creates:

```
alembic.ini
migrations/
├── env.py
├── script.py.mako
└── versions/          <- revision files land here
```

### `models.py` — the schema written in Python

Alembic reads this file to know what the schema *should* look like. `__tablename__` becomes the table, each `Column` becomes a column.

```python
# app/models.py
import sqlalchemy as sa
from sqlalchemy.orm import DeclarativeBase

NAMING = {
    "pk": "pk_%(table_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "ix": "ix_%(table_name)s_%(column_0_name)s",
}


class Base(DeclarativeBase):
    metadata = sa.MetaData(naming_convention=NAMING)


class Book(Base):
    __tablename__ = "books"

    book_id = sa.Column(sa.String(16), primary_key=True)
    title = sa.Column(sa.Text, nullable=False)
    author = sa.Column(sa.Text, nullable=False)
    isbn = sa.Column(sa.Text, unique=True)
    created_at = sa.Column(sa.DateTime(timezone=True), server_default=sa.func.now(), nullable=False)


class Member(Base):
    __tablename__ = "members"

    member_id = sa.Column(sa.String(16), primary_key=True)
    full_name = sa.Column(sa.Text, nullable=False)
    email = sa.Column(sa.Text, nullable=False, unique=True)
    joined_on = sa.Column(sa.Date, server_default=sa.func.current_date(), nullable=False)


class Copy(Base):
    __tablename__ = "copies"

    copy_id = sa.Column(sa.Integer, primary_key=True)
    book_id = sa.Column(sa.String(16), sa.ForeignKey("books.book_id"), nullable=False)
    shelf = sa.Column(sa.Text, nullable=False)
    status = sa.Column(sa.Text, nullable=False, server_default="available")
    __table_args__ = (
        sa.CheckConstraint("status IN ('available', 'on_loan', 'lost')", name="status_valid"),
    )


class Loan(Base):
    __tablename__ = "loans"

    loan_id = sa.Column(sa.Integer, primary_key=True)
    copy_id = sa.Column(sa.Integer, sa.ForeignKey("copies.copy_id"), nullable=False)
    member_id = sa.Column(sa.String(16), sa.ForeignKey("members.member_id"), nullable=False)
    borrowed_on = sa.Column(sa.Date, server_default=sa.func.current_date(), nullable=False)
    due_on = sa.Column(sa.Date, nullable=False)
    returned_on = sa.Column(sa.Date)
    __table_args__ = (
        sa.CheckConstraint("due_on >= borrowed_on", name="due_after_borrow"),
        sa.Index("one_open_loan_per_copy", "copy_id", unique=True,
                 postgresql_where=sa.text("returned_on IS NULL")),
    )


class Fine(Base):
    __tablename__ = "fines"

    fine_id = sa.Column(sa.Integer, primary_key=True)
    loan_id = sa.Column(sa.Integer, sa.ForeignKey("loans.loan_id"), nullable=False)
    amount_cents = sa.Column(sa.Integer, nullable=False)
    paid_on = sa.Column(sa.Date)
    __table_args__ = (
        sa.CheckConstraint("amount_cents >= 0", name="amount_not_negative"),
    )
```

### `env.py` — the three lines that matter

Everything else in this file is already correct. Add these.

```python
# migrations/env.py
import os
import sys

sys.path.append(os.getcwd())         # 0. so `app` is importable

from app.models import Base          # 1. your models

target_metadata = Base.metadata      # 2. what autogenerate compares against

config.set_main_option("sqlalchemy.url", DATABASE_URL)   # 3. your database
```

`DATABASE_URL` here is the same string from the top of this notebook.

### The first migration

```bash
$ alembic revision --autogenerate -m 'library schema v1'
INFO  [alembic.autogenerate.compare] Detected added table 'books'
INFO  [alembic.autogenerate.compare] Detected added table 'members'
INFO  [alembic.autogenerate.compare] Detected added table 'copies'
INFO  [alembic.autogenerate.compare] Detected added table 'loans'
INFO  [alembic.autogenerate.compare] Detected added table 'fines'
INFO  [alembic.autogenerate.compare] Detected added index 'one_open_loan_per_copy' on '()'
Generating migrations/versions/a72317edfb3a_library_schema_v1.py ... done
```

Alembic wrote the file. You do not write `CREATE TABLE` by hand any more — this is what the notebook did in cells above, captured as code:

```python
# migrations/versions/a72317edfb3a_library_schema_v1.py
revision = 'a72317edfb3a'
down_revision = None


def upgrade() -> None:
    op.create_table('books',
        sa.Column('book_id', sa.String(length=16), nullable=False),
        sa.Column('title', sa.Text(), nullable=False),
        sa.Column('author', sa.Text(), nullable=False),
        sa.Column('isbn', sa.Text(), nullable=True),
        sa.Column('created_at', sa.DateTime(timezone=True),
                  server_default=sa.text('now()'), nullable=False),
        sa.PrimaryKeyConstraint('book_id', name=op.f('pk_books')),
        sa.UniqueConstraint('isbn', name=op.f('uq_books_isbn')),
    )
    # ... members, copies, loans, fines ...
    op.create_index('one_open_loan_per_copy', 'loans', ['copy_id'], unique=True,
                    postgresql_where=sa.text('returned_on IS NULL'))


def downgrade() -> None:
    op.drop_index('one_open_loan_per_copy', table_name='loans',
                  postgresql_where=sa.text('returned_on IS NULL'))
    op.drop_table('fines')
    op.drop_table('loans')
    op.drop_table('copies')
    op.drop_table('members')
    op.drop_table('books')
```

`upgrade` builds the schema, `downgrade` takes it apart — in reverse order, because of the foreign keys.

### The pieces

| Piece | What it is |
|---|---|
| `alembic.ini` | config. Points Alembic at `migrations/`. |
| `migrations/env.py` | runs on every command. Must know your models and your database. |
| `migrations/versions/a72317edfb3a_library_schema_v1.py` | one revision: an id, a parent id, `upgrade()`, `downgrade()` |
| `alembic_version` | a one-row table Alembic puts in your database to record which revision is applied |

| Command | What it does |
|---|---|
| `alembic init migrations` | creates the folder. Once per project. |
| `alembic revision --autogenerate -m "..."` | writes a new revision from the difference between models and database |
| `alembic upgrade head` | applies everything not yet applied |
| `alembic downgrade -1` | undoes the last revision |
| `alembic current` | which revision is applied now |
| `alembic history` | all revisions, in order |

The migration file is the source of truth, not the models. `autogenerate` writes it once; after that you edit the file and commit it like any other code.

In [ ]:
try:
    revision, = q("SELECT version_num FROM alembic_version", fetch=True)[0]
    print("database is on revision:", revision)
except Exception:
    print("no alembic_version table yet.")
    print("Run: alembic upgrade head")

# and confirm the partial index kept its WHERE clause
try:
    ddl, = q("SELECT indexdef FROM pg_indexes WHERE indexname = 'one_open_loan_per_copy'", fetch=True)[0]
    print(ddl)
except Exception:
    print("index check needs alembic upgrade head first")

### Changing the schema later

A phone number for members. You change `models.py`, then ask Alembic what changed:

```bash
$ alembic revision --autogenerate -m 'add member phone'
INFO  [alembic.autogenerate.compare] Detected added column 'members.phone'
Generating migrations/versions/377cee7648d2_add_member_phone.py ... done
```

```python
# migrations/versions/377cee7648d2_add_member_phone.py
revision = '377cee7648d2'
down_revision = 'a72317edfb3a'      # the previous revision — this is the chain


def upgrade() -> None:
    op.add_column('members', sa.Column('phone', sa.Text(), nullable=True))


def downgrade() -> None:
    op.drop_column('members', 'phone')
```

`down_revision` is what makes it a history rather than a pile of files. `upgrade head` walks the chain from wherever the database is.

| Situation | The right command |
|---|---|
| Fresh database | `alembic upgrade head` |
| Database on an older revision | `alembic upgrade head` |
| Bad deploy | `alembic downgrade -1` |
| Someone edited a migration already applied | Write a new revision. Never edit history. |

## Harbour schema v1

Two tables. A **document** is a source file; a **chunk** is a piece of it, small enough to search. One document becomes many chunks — the same shape as book and copy.

The `embedding` column comes in week 10 (it needs the pgvector extension), so it is not here yet. That is exactly the kind of change a migration exists for.

In [ ]:
q("DROP TABLE IF EXISTS chunks CASCADE")
q("DROP TABLE IF EXISTS documents CASCADE")

q("""
CREATE TABLE documents (
    doc_id          SERIAL PRIMARY KEY,
    title           TEXT NOT NULL,
    source_path     TEXT NOT NULL,
    content_sha256  TEXT NOT NULL UNIQUE,
    ingested_at     TIMESTAMPTZ NOT NULL DEFAULT now()
)
""")

q("""
CREATE TABLE chunks (
    chunk_id  SERIAL PRIMARY KEY,
    doc_id    INTEGER NOT NULL REFERENCES documents(doc_id) ON DELETE CASCADE,
    ordinal   INTEGER NOT NULL,
    content   TEXT NOT NULL,
    UNIQUE (doc_id, ordinal)
)
""")

print("documents and chunks created")

## Harbour schema v1

### documents

| Column | Type | Rule |
|---|---|---|
| `doc_id` | SERIAL | primary key |
| `title` | TEXT | not null |
| `source_path` | TEXT | where the file came from |
| `content_sha256` | TEXT | unique, not null |
| `ingested_at` | TIMESTAMPTZ | set by the database |

### chunks

| Column | Type | Rule |
|---|---|---|
| `chunk_id` | SERIAL | primary key |
| `doc_id` | INTEGER | foreign key -> `documents.doc_id`, deleted with the document |
| `ordinal` | INTEGER | position inside the document |
| `content` | TEXT | the passage itself |
| *(later)* `embedding` | `vector(384)` | added in week 10 with pgvector |

`UNIQUE (doc_id, ordinal)` is two columns acting as one key: a document cannot have two "chunk 3"s.

### Why `content_sha256` is unique

Ingesting the same file twice should not create two documents. The hash of the content is the identity — the same file always hashes the same, so the database rejects the second copy. That is week 2's idempotency rule, enforced by the schema instead of by application code.

### How they connect

| From | To | Relationship |
|---|---|---|
| `documents` | `chunks` | one document, many chunks |

## Side by side

| | Library | Harbour |
|---|---|---|
| Versioned with | Alembic | Alembic, same commands |
| Identity | `book_id` TEXT, readable (`BK-001`) | `doc_id` SERIAL, database-assigned |
| Parent / child | `books` -> `copies` | `documents` -> `chunks` |
| Uniqueness rule | one open loan per copy (partial index) | one row per content hash |
| Money | `amount_cents` INTEGER | — |
| Changes later | `members.phone` | `chunks.embedding vector(384)` |

Every schema change is one revision file with an id and a parent. That is the whole habit.